# From 189 to 242: ChEMBL against Pharmaprojects

The title refers to two counts of approved, genetically supported target-indication pairs: 189
reported by Minikel et al. 2024 from Pharmaprojects, and 242 in this paper from ChEMBL.

Results 6 says that 242 approved target-indication pairs carry GWAS support, "a substantial
increase (28%) from the 189 pairs recently reported by Minikel et al.", and attributes the
increase to more GWAS, richer functional genomics, a better gene-prioritisation model *and* the
larger ChEMBL resource. Those are two changes at once: the genetic evidence and the drug
resource. This notebook separates them by changing one input at a time:

| step | genetic support | drug resource |
| --- | --- | --- |
| Minikel et al. | their `geneticSupport_old` (MeSH similarity >= 0.8, OTG v8 / OMIM / PICCOLO / Genebass / intOGen) | Pharmaprojects |
| swap the evidence | ours, `score_all` (the definition that gives 242) | Pharmaprojects |
| swap the resource | ours, `score_all` | ChEMBL |

Control step first: (a) the 242 from the current pipeline, (b) launched Pharmaprojects pairs and
the 447 shared with ChEMBL Phase IV quoted in Supplementary Results 11.4, (c) the launched pairs
carrying Pharmaprojects' own Minikel-era support flag, against 189. Nothing here is tuned to hit
a published value.

Then: (1) launched Pharmaprojects pairs with our support, (2) pairs by maximum phase and unique
targets in each resource, (3) overlap of approved pairs, and (4) whether the 242 pairs missing
from Pharmaprojects are older drugs.

Inputs: `ti_pairs_chembl` and `ti_pairs_pharmaprojects` from `01-data-preparation/12`, the 25.06
ChEMBL evidence, and the 25.06 `drug_molecule` index for approval years (fetched below if
absent). The Pharmaprojects data are the subset that Minikel et al. 2024 released publicly
(https://github.com/ericminikel/genetic_support, CC BY 4.0, with Citeline's permission); they are
not part of the Zenodo deposit. Their original table, `combined_ti_TA.tsv`, is no longer in the
working tree; the funnel in Control (c) reads it from this repository's git history when that is
available and is skipped otherwise.

In [1]:
import subprocess
import urllib.request
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from scipy.stats import mannwhitneyu

from manuscript_methods import enrichment, paper

numbers = {}

chembl = pd.read_parquet(paper.derived("ti_pairs_chembl"))
pharmaprojects = pd.read_parquet(paper.derived("ti_pairs_pharmaprojects"))
for frame in (chembl, pharmaprojects):
    frame["support"] = enrichment.support_mask(frame)

disease_index = pd.read_parquet(paper.release("disease") + "/disease.parquet", columns=["id", "ancestors"])
ancestors = disease_index.set_index("id")["ancestors"].apply(set)
# Same oncology set as `chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list`.
oncology = {term for term, parents in ancestors.items() if "MONDO_0045024" in parents} | {"MONDO_0045024"}
print(f"ChEMBL pairs {len(chembl):,} | Pharmaprojects rows {len(pharmaprojects):,} | oncology terms {len(oncology):,}")


def pairs(frame):
    # (targetId, diseaseId) set
    return set(zip(frame["targetId"], frame["diseaseId"]))

ChEMBL pairs 37,377 | Pharmaprojects rows 7,390 | oncology terms 3,593


## Control (a): the 242 approved GWAS-supported ChEMBL pairs

Support is `score_all` present: an L2G-prioritised gene (credible-set L2G >= 0.1) for the
indication or any of its ontology descendants. Approved is ChEMBL maximum phase 4.

In [2]:
control = enrichment.or_rs(chembl["support"], chembl["approved"])
numbers["control_chembl_approved_supported"] = control["yes_evid-high_clinphase"]
numbers["control_chembl_or"] = round(control["odds_ratio"], 4)

published = pd.read_csv(paper.baseline("242_approved_TI_pairs.csv"), sep="\t", index_col=0)
ours = pairs(chembl[chembl["support"] & (chembl["approved"] == 1)])
numbers["control_242_set_agreement"] = len(ours & pairs(published))
print(f"approved and supported: {control['yes_evid-high_clinphase']} (OR {control['odds_ratio']:.4f})")
print(f"identical to 242_approved_TI_pairs.csv on {numbers['control_242_set_agreement']} of {len(published)} pairs")

approved and supported: 242 (OR 3.6186)
identical to 242_approved_TI_pairs.csv on 242 of 242 pairs


## Control (b): launched Pharmaprojects pairs and the 447

The table is Minikel et al.'s T-I list after MeSH -> ontology mapping, which was exploded, so a
row is a (target, ontology term) pair and one Pharmaprojects T-I pair (`ti_uid`, gene-MeSH) can
occupy several rows. The three units are kept apart below.

In [3]:
launched = pharmaprojects[pharmaprojects["approved"] == 1]
numbers["pp_launched_rows"] = len(launched)
numbers["pp_launched_target_disease"] = len(pairs(launched))
numbers["pp_launched_ti_uid"] = launched["ti_uid"].nunique()

phase_four = pairs(chembl[chembl["approved"] == 1])
numbers["pp_launched_shared_chembl_iv"] = len(pairs(launched) & phase_four)
shared_ti = (
    launched.assign(shared=[pair in phase_four for pair in zip(launched["targetId"], launched["diseaseId"])])
    .groupby("ti_uid")["shared"]
    .any()
)
numbers["pp_launched_ti_uid_shared_chembl_iv"] = int(shared_ti.sum())

print(
    f"launched: {numbers['pp_launched_rows']} rows = {numbers['pp_launched_target_disease']} (target, disease) pairs "
    f"= {numbers['pp_launched_ti_uid']} gene-MeSH pairs"
)
print(
    f"also ChEMBL Phase IV: {numbers['pp_launched_shared_chembl_iv']} of {numbers['pp_launched_target_disease']} "
    f"(target, disease) pairs ({100 * numbers['pp_launched_shared_chembl_iv'] / numbers['pp_launched_target_disease']:.1f}%); "
    f"{numbers['pp_launched_ti_uid_shared_chembl_iv']} of {numbers['pp_launched_ti_uid']} gene-MeSH pairs"
)

launched: 913 rows = 911 (target, disease) pairs = 822 gene-MeSH pairs
also ChEMBL Phase IV: 447 of 911 (target, disease) pairs (49.1%); 429 of 822 gene-MeSH pairs


447 reproduces, but Supplementary Results 11.4 divides it by 913 (rows) where the numerator
counts distinct (target, disease) pairs, of which there are 911. Both round to 49%. Counted as
Pharmaprojects' own T-I pairs, it is 429 of 822 (52%).

## Control (c): Minikel-era support among launched pairs, against 189

`geneticSupport_old` is Minikel et al.'s `target_status == "genetically supported target"`,
which is MeSH similarity >= 0.8 between the indication and an associated trait (the column
equals `similarity >= 0.8` row for row). The funnel follows the steps that built the table
(in the original analysis, not public): their table, MeSH ->
ontology mapping through `dbXRefs` / `obsoleteXRefs`, explode, drop oncology, keep Phase I and
later.

In [4]:
supported_launched = launched[launched["geneticSupport_old"] == 1]
numbers["minikel_flag_launched_rows"] = len(supported_launched)
numbers["minikel_flag_launched_ti_uid"] = supported_launched["ti_uid"].nunique()
print(
    f"in our table: {numbers['minikel_flag_launched_rows']} rows, "
    f"{numbers['minikel_flag_launched_ti_uid']} gene-MeSH pairs"
)
print(supported_launched.drop_duplicates("ti_uid")["genetic_insight"].value_counts().to_string())

# Minikel et al.'s original table, read from this repository's git history (no longer in the tree).
try:
    raw = subprocess.run(
        ["git", "show", "dcdc3ce^:chapters/_legacy/05-other-drug-indication-data/combined_ti_TA.tsv"],
        capture_output=True,
        text=True,
        check=True,
        cwd=paper.ROOT,
    ).stdout
    original = pd.read_csv(StringIO(raw), sep="\t", low_memory=False)
except subprocess.CalledProcessError:
    original = None
    print("combined_ti_TA.tsv not available; funnel skipped")

in our table: 192 rows, 170 gene-MeSH pairs
genetic_insight
both         137
gwas          21
omim/tcga     12


In [5]:
if original is not None:
    xrefs = pd.read_parquet(paper.release("disease") + "/disease.parquet", columns=["id", "dbXRefs", "obsoleteXRefs"])
    mesh = []
    for column in ["dbXRefs", "obsoleteXRefs"]:
        exploded = xrefs[["id", column]].explode(column).dropna()
        exploded["meshId"] = exploded[column].str.upper().str.extract(r"MESH:\s*([^,\s]+)")[0]
        mesh.append(exploded[["id", "meshId"]].dropna())
    mesh_to_terms = pd.concat(mesh).drop_duplicates().groupby("meshId")["id"].apply(list)

    original["terms"] = original["indication_mesh_id"].str.upper().map(mesh_to_terms)
    original["mapped"] = original["terms"].notna()
    original["oncology_only"] = original["terms"].apply(
        lambda terms: isinstance(terms, list) and all(term in oncology for term in terms)
    )
    flagged = original[(original["ccat"] == "Launched") & (original["target_status"] == "genetically supported target")]
    funnel = pd.Series(
        {
            "Minikel et al. table, launched with support": len(flagged),
            "  of which germline (not intOGen)": int((flagged["assoc_source"] != "intOGen").sum()),
            "  of which MeSH maps to the ontology": int(flagged["mapped"].sum()),
            "  of which mapped and not oncology": int((flagged["mapped"] & ~flagged["oncology_only"]).sum()),
            "  in ti_pairs_pharmaprojects (gene-MeSH pairs)": numbers["minikel_flag_launched_ti_uid"],
            "  in ti_pairs_pharmaprojects (rows)": numbers["minikel_flag_launched_rows"],
        }
    )
    numbers["minikel_table_launched_supported"] = int(funnel.iloc[0])
    numbers["minikel_table_launched_supported_germline"] = int(funnel.iloc[1])
    numbers["minikel_table_launched_supported_mapped_non_oncology"] = int(funnel.iloc[3])
    print(funnel.to_string())
    print()
    print("best-association source, mapped and not oncology:")
    print(flagged[flagged["mapped"] & ~flagged["oncology_only"]]["assoc_source"].value_counts().to_string())

Minikel et al. table, launched with support       278
  of which germline (not intOGen)                 211
  of which MeSH maps to the ontology              245
  of which mapped and not oncology                171
  in ti_pairs_pharmaprojects (gene-MeSH pairs)    170
  in ti_pairs_pharmaprojects (rows)               192

best-association source, mapped and not oncology:
assoc_source
OTG         106
OMIM         59
Genebass      4
PICCOLO       2


**No step gives 189.** The Minikel et al. table has 278 launched supported T-I pairs, 211 of them
germline; after mapping and oncology removal 171 remain, and 170 survive into our table (one
loses its gene symbol in the Ensembl join), 192 rows once exploded. Nothing in either repository
computes 189, and Minikel et al.'s main text does not state it. It may come from a figure or
supplementary table of theirs, at a filter this table does not carry (programme start year,
all-time launches, oncology kept), so it is left as unverified rather than matched.

## 1. Launched Pharmaprojects pairs with our genetic support

Same definition as the 242: `score_all` present for the (target, disease) pair.

How it differs from Minikel et al.'s definition:

- **Evidence.** Ours is Gentropy GWAS credible sets in the 25.06 release with L2G >= 0.1 for the
  gene, any credible set. Theirs pools OTG v8 GWAS (L2G *share* >= 0.5, i.e. the top gene),
  OMIM, PICCOLO colocalisation, Genebass exome burden and intOGen somatic drivers.
- **Indication matching.** Ours propagates up the EFO/MONDO ontology: a pair is supported when
  the indication *or any descendant* has an association. Theirs uses MeSH Lin/Resnik similarity
  >= 0.8 between indication and trait, which also matches siblings and ancestors.
- **Universe.** Both are evaluated here on the same 7,390 rows, which are already restricted to
  their "genetic insight" indications, so the denominator does not change between steps.

In [6]:
numbers["ours_launched_rows"] = int(launched["support"].sum())
numbers["ours_launched_ti_uid"] = launched.loc[launched["support"], "ti_uid"].nunique()
numbers["ours_launched_target_disease"] = len(pairs(launched[launched["support"]]))
numbers["both_flags_launched_ti_uid"] = launched.loc[
    launched["support"] & (launched["geneticSupport_old"] == 1), "ti_uid"
].nunique()
numbers["ours_launched_shared_chembl_iv"] = len(pairs(launched[launched["support"]]) & phase_four)

pp_ours = enrichment.or_rs(pharmaprojects["support"], pharmaprojects["approved"])
pp_theirs = enrichment.or_rs(pharmaprojects["geneticSupport_old"] == 1, pharmaprojects["approved"])
print(
    f"our support, launched: {numbers['ours_launched_rows']} rows = {numbers['ours_launched_target_disease']} "
    f"(target, disease) = {numbers['ours_launched_ti_uid']} gene-MeSH pairs; OR {pp_ours['odds_ratio']:.2f}"
)
print(
    f"their support, launched: {numbers['minikel_flag_launched_rows']} rows = "
    f"{numbers['minikel_flag_launched_ti_uid']} gene-MeSH pairs; OR {pp_theirs['odds_ratio']:.2f}"
)
print(f"supported by both: {numbers['both_flags_launched_ti_uid']} gene-MeSH pairs")
print(f"our launched supported pairs that are also ChEMBL Phase IV: {numbers['ours_launched_shared_chembl_iv']}")

our support, launched: 86 rows = 86 (target, disease) = 83 gene-MeSH pairs; OR 1.65
their support, launched: 192 rows = 170 gene-MeSH pairs; OR 2.32
supported by both: 64 gene-MeSH pairs
our launched supported pairs that are also ChEMBL Phase IV: 56


## 2. Pairs by maximum phase, and unique targets

The two universes are not the same: ChEMBL is every non-oncology T-I pair at Phase I or later;
the Pharmaprojects table is Minikel et al.'s programmes added since 2000, restricted to
indications with genetic insight and with a MeSH term that maps to the ontology.

In [7]:
def by_phase(frame, unit):
    counts = frame.groupby(unit)["maxClinicalPhase"].max().astype(int).value_counts().sort_index()
    return counts.reindex([1, 2, 3, 4], fill_value=0)


phase_table = pd.DataFrame(
    {
        "ChEMBL (target, disease)": by_phase(chembl, ["targetId", "diseaseId"]),
        "Pharmaprojects (target, disease)": by_phase(pharmaprojects, ["targetId", "diseaseId"]),
        "Pharmaprojects gene-MeSH": by_phase(pharmaprojects, ["ti_uid"]),
    }
)
phase_table.index = ["Phase I", "Phase II", "Phase III", "Phase IV / launched"]
phase_table.loc["total"] = phase_table.sum()

targets = pd.DataFrame(
    {
        "ChEMBL": [chembl.loc[chembl["maxClinicalPhase"] == p, "targetId"].nunique() for p in [1, 2, 3, 4]]
        + [chembl["targetId"].nunique()],
        "Pharmaprojects": [
            pharmaprojects.loc[pharmaprojects["maxClinicalPhase"] == p, "targetId"].nunique() for p in [1, 2, 3, 4]
        ]
        + [pharmaprojects["targetId"].nunique()],
    },
    index=phase_table.index,
)
for resource, frame in [("chembl", chembl), ("pp", pharmaprojects)]:
    numbers[f"{resource}_targets"] = frame["targetId"].nunique()
    numbers[f"{resource}_diseases"] = frame["diseaseId"].nunique()
for column, key in [("ChEMBL (target, disease)", "chembl"), ("Pharmaprojects (target, disease)", "pp")]:
    for phase, value in zip([1, 2, 3, 4], phase_table[column].iloc[:4]):
        numbers[f"{key}_pairs_phase_{phase}"] = int(value)
print(phase_table.to_string())
print()
print("unique targets with a pair at that maximum phase:")
print(targets.to_string())
print()
print(f"distinct indications: ChEMBL {numbers['chembl_diseases']:,}, Pharmaprojects {numbers['pp_diseases']:,}")

                     ChEMBL (target, disease)  Pharmaprojects (target, disease)  Pharmaprojects gene-MeSH
Phase I                                  6163                              2255                      1947
Phase II                                14410                              3301                      2986
Phase III                               12240                               900                       822
Phase IV / launched                      4564                               911                       822
total                                   37377                              7367                      6577

unique targets with a pair at that maximum phase:
                     ChEMBL  Pharmaprojects
Phase I                 976             733
Phase II               1131             844
Phase III               903             366
Phase IV / launched     754             319
total                  1273            1164

distinct indications: ChEMBL 1,842, Pharmaprojec

## 3. Overlap of approved pairs

Matched on exact (target, ontology term). Pharmaprojects-only pairs are then looked up in
ChEMBL at any phase.

In [8]:
pp_launched = pairs(launched)
numbers["approved_both"] = len(phase_four & pp_launched)
numbers["approved_chembl_only"] = len(phase_four - pp_launched)
numbers["approved_pp_only"] = len(pp_launched - phase_four)
chembl_phase = chembl.set_index(["targetId", "diseaseId"])["maxClinicalPhase"]
pp_only_phase = pd.Series([chembl_phase.get(pair, np.nan) for pair in pp_launched - phase_four])
print(
    f"approved in both {numbers['approved_both']:,} | ChEMBL only {numbers['approved_chembl_only']:,} | "
    f"Pharmaprojects only {numbers['approved_pp_only']:,}"
)
print("Pharmaprojects-only launched pairs, ChEMBL maximum phase:")
print(
    pp_only_phase.map(lambda phase: "absent" if np.isnan(phase) else f"Phase {int(phase)}").value_counts().to_string()
)

approved in both 447 | ChEMBL only 4,117 | Pharmaprojects only 464
Pharmaprojects-only launched pairs, ChEMBL maximum phase:
absent     273
Phase 3    155
Phase 2     31
Phase 1      5


## 4. The 242 pairs absent from Pharmaprojects, and their approval years

Approval year of a pair is the earliest `yearOfFirstApproval` among the drugs with a Phase IV
ChEMBL record for that (target, disease), salt forms falling back to their parent molecule.
`yearOfFirstApproval` is the drug's first approval for any indication, so this measures drug age,
which is what the "older drugs" explanation is about.

"Absent" is tested at three levels: not launched in the Pharmaprojects table; not launched even
after allowing the indication to be an ontology ancestor or descendant of a launched
Pharmaprojects indication for the same target; and not present at any phase in Minikel et al.'s
full post-2000 T-I list (29,481 rows, before the genetic-insight filter).

In [9]:
molecule_path = Path(paper.release("drug_molecule"))
if not molecule_path.exists():
    # Public, same 25.06 release as the rest.
    base = "https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/25.06/output/drug_molecule/"
    part = "part-00000-8418c096-e923-475d-a569-723fe91bc963-c000.snappy.parquet"
    molecule_path.mkdir(parents=True)
    urllib.request.urlretrieve(base + part, molecule_path / part)
molecules = pd.read_parquet(molecule_path, columns=["id", "parentId", "yearOfFirstApproval"]).set_index("id")

evidence = (
    ds.dataset(paper.release("evidence") + "/sourceId=chembl")
    .to_table(columns=["targetId", "diseaseId", "drugId", "clinicalPhase"])
    .to_pandas()
)
evidence = evidence[(evidence["clinicalPhase"] == 4) & ~evidence["diseaseId"].isin(oncology)].copy()
evidence["year"] = evidence["drugId"].map(molecules["yearOfFirstApproval"])
missing = evidence["year"].isna()
evidence.loc[missing, "year"] = (
    evidence.loc[missing, "drugId"].map(molecules["parentId"]).map(molecules["yearOfFirstApproval"])
)
pair_year = evidence.groupby(["targetId", "diseaseId"])["year"].min()

supported = chembl[chembl["support"] & (chembl["approved"] == 1)][["targetId", "diseaseId"]].copy()
keys = list(zip(supported["targetId"], supported["diseaseId"]))
supported["year"] = [pair_year.get(pair, np.nan) for pair in keys]
numbers["supported_with_year"] = int(supported["year"].notna().sum())
print(f"{numbers['supported_with_year']} of {len(supported)} pairs have an approval year")

236 of 242 pairs have an approval year


In [10]:
full = pd.read_parquet(paper.baseline("minikel_etal_data_processed.parquet"))
full = full.explode("targetIds").explode("diseaseIds").dropna(subset=["targetIds", "diseaseIds"])
full_pairs = set(zip(full["targetIds"], full["diseaseIds"]))

launched_terms = launched.groupby("targetId")["diseaseId"].apply(set)


def related_launch(target, disease):
    # A launched Pharmaprojects indication for the target is the term, an ancestor or a descendant.
    if target not in launched_terms:
        return False
    above = ancestors.get(disease, set())
    return any(
        term == disease or term in above or disease in ancestors.get(term, set()) for term in launched_terms[target]
    )


supported["pp_launched"] = [pair in pp_launched for pair in keys]
supported["pp_launched_related"] = [related_launch(*pair) for pair in keys]
supported["pp_any_phase"] = [pair in pairs(pharmaprojects) for pair in keys]
supported["full_list_any_phase"] = [pair in full_pairs for pair in keys]

rows = []
for column, label in [
    ("pp_launched", "launched in the table, exact term"),
    ("pp_launched_related", "launched in the table, related term"),
    ("full_list_any_phase", "anywhere in the full post-2000 list"),
]:
    shared = supported.loc[supported[column], "year"].dropna()
    absent = supported.loc[~supported[column], "year"].dropna()
    rows.append(
        {
            "shared with Pharmaprojects": label,
            "absent": int((~supported[column]).sum()),
            "shared": int(supported[column].sum()),
            "median year absent": absent.median(),
            "median year shared": shared.median(),
            "% before 2000 absent": round(100 * (absent < 2000).mean(), 1),
            "% before 2000 shared": round(100 * (shared < 2000).mean(), 1),
            "Mann-Whitney P": mannwhitneyu(absent, shared).pvalue,
        }
    )
absence = pd.DataFrame(rows).set_index("shared with Pharmaprojects")
numbers["absent_from_pp_launched"] = int(absence.iloc[0]["absent"])
numbers["absent_median_year"] = float(absence.iloc[0]["median year absent"])
numbers["shared_median_year"] = float(absence.iloc[0]["median year shared"])
numbers["absent_pct_pre2000"] = float(absence.iloc[0]["% before 2000 absent"])
numbers["shared_pct_pre2000"] = float(absence.iloc[0]["% before 2000 shared"])
numbers["absent_vs_shared_mwu_p"] = float(absence.iloc[0]["Mann-Whitney P"])
numbers["absent_related_term"] = int(absence.iloc[1]["absent"])
numbers["absent_full_list"] = int(absence.iloc[2]["absent"])
absence.round(3)

,absent,shared,median year absent,median year shared,% before 2000 absent,% before 2000 shared,Mann-Whitney P
shared with Pharmaprojects,,,,,,,
"launched in the table, exact term",186,56,2004.0,2001.5,43.9,50.0,0.311
"launched in the table, related term",129,113,2003.0,2004.0,43.9,46.9,0.509
anywhere in the full post-2000 list,128,114,2001.5,2004.5,47.5,43.0,0.755


In [11]:
decades = pd.cut(
    supported["year"],
    [0, 1980, 1990, 2000, 2010, 2030],
    right=False,
    labels=["before 1980", "1980s", "1990s", "2000s", "2010 on"],
)
print(pd.crosstab(decades, supported["pp_launched"].map({True: "shared", False: "absent"}), dropna=False).to_string())

pp_launched  absent  shared
year                       
NaN               6       0
before 1980      37      20
1980s            22       4
1990s            20       4
2000s            31      12
2010 on          70      16


Why the 186 are absent, by what the Pharmaprojects table is missing:

In [12]:
absent = supported[~supported["pp_launched"]].copy()
pp_targets, pp_terms = set(pharmaprojects["targetId"]), set(pharmaprojects["diseaseId"])
absent["reason"] = np.select(
    [
        ~absent["targetId"].isin(pp_targets) & ~absent["diseaseId"].isin(pp_terms),
        ~absent["targetId"].isin(pp_targets),
        ~absent["diseaseId"].isin(pp_terms),
        absent["pp_any_phase"],
    ],
    [
        "neither target nor indication in the table",
        "target not in the table",
        "indication not in the table",
        "pair in the table, below launch",
    ],
    default="target and indication both present, never paired",
)
reasons = absent.groupby("reason").agg(pairs=("year", "size"), median_year=("year", "median"))
reasons.sort_values("pairs", ascending=False)

,pairs,median_year
reason,,
"target and indication both present, never paired",71,1997.0
target not in the table,39,2010.0
indication not in the table,37,1998.0
"pair in the table, below launch",30,2009.5
neither target nor indication in the table,9,2010.0


**The "older drugs" explanation does not hold.** The 186 supported approved pairs that
Pharmaprojects does not have launched are, if anything, slightly *newer* than the 56 it shares
(median first approval 2004 against 2001.5, 44% against 50% approved before 2000, P = 0.31). The
same holds under ontology-relaxed matching and against the full post-2000 list. The groups that
are absent because Pharmaprojects lacks the target, or has the pair only below launch, are the
youngest (median 2009-2010). What separates the resources is coverage of indications and targets
(ChEMBL has 1,842 indications and 4,564 approved pairs, the table 523 and 911), not drug age.

## Summary

Changing one input at a time, the increase does not come from the genetic evidence. On the same
Pharmaprojects pairs our support marks **fewer** launched pairs than Minikel et al.'s own flag
(83 against 170 gene-MeSH pairs, with 64 in common). The whole increase, and more, comes from
changing the drug resource: ChEMBL has five times as many approved non-oncology pairs (4,564
against 911), and 186 of the 242 are not launched pairs in Pharmaprojects.

In [13]:
steps = pd.DataFrame(
    [
        ("Minikel et al. (published)", "theirs", "Pharmaprojects", 189, "not reproduced"),
        (
            "their flag, this table",
            "theirs",
            "Pharmaprojects",
            numbers["minikel_flag_launched_ti_uid"],
            f"{numbers['minikel_flag_launched_rows']} rows",
        ),
        (
            "swap the evidence",
            "ours (score_all)",
            "Pharmaprojects",
            numbers["ours_launched_ti_uid"],
            f"{numbers['ours_launched_rows']} rows",
        ),
        (
            "swap the resource",
            "ours (score_all)",
            "ChEMBL",
            numbers["control_chembl_approved_supported"],
            "(target, disease) pairs",
        ),
    ],
    columns=["step", "support", "resource", "approved supported pairs", "note"],
)
print(steps.to_string(index=False))
print(paper.save_results("chembl_vs_pharmaprojects", numbers))

                      step          support       resource  approved supported pairs                    note
Minikel et al. (published)           theirs Pharmaprojects                       189          not reproduced
    their flag, this table           theirs Pharmaprojects                       170                192 rows
         swap the evidence ours (score_all) Pharmaprojects                        83                 86 rows
         swap the resource ours (score_all)         ChEMBL                       242 (target, disease) pairs
/Users/yt4/Projects/Gentropy-manuscript/results/chembl_vs_pharmaprojects.json
